# Random Protein Sampling

![Random Protein Sampling](https://proto-bio.github.io/proto-assets/images/tool/random_protein/hero.png)

This notebook demonstrates `run_random_protein_sample`, which generates protein sequence variants by substituting amino acids at masked positions using codon scheme-biased distributions. It covers pre-masked sequences, automatic position selection via masking strategies, and codon scheme comparisons that mirror the amino acid distributions of common degenerate oligonucleotide libraries.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("random_protein")
display_overview("random_protein")
display_docs_section("random_protein", "Background")

# Random Protein Sampling

Random Protein Sampling fills masked positions in protein sequences with amino acids drawn from a codon scheme. Positions can be marked directly with `_` or selected automatically by a masking strategy. The codon scheme sets amino-acid frequencies: `UNIFORM` weights all twenty equally, while degenerate schemes such as `NNK` or `NDT` weight each amino acid by how many codons encode it. Optional amino-acid exclusions remove unwanted residues such as cysteine. It runs on CPU with no model and no external dependencies.

Random Protein Sampling performs random mutagenesis at the protein level: it takes a protein sequence, determines which positions are designable, and replaces each with an amino acid sampled from the distribution implied by a codon scheme. It generates protein-sequence diversity without any learned model, the simplest possible baseline against which model-guided designers can be compared.

Internally, designable positions are either the `_` characters already present in the input or, when none are present, positions chosen by the configured masking strategy. The codon scheme is expanded to its concrete codons, and each amino acid's sampling weight is set proportional to the number of codons in the scheme that encode it, with stop codons excluded by default. `UNIFORM` instead assigns equal weight to all twenty standard amino acids. If `excluded_amino_acids` is set, those residue types are removed after codon-scheme and stop-codon handling; the sampler raises an error if no reachable residue remains. Each masked position is filled independently by a weighted random draw. With a fixed seed the output is deterministic.

This tool is original proto-tools code maintained by [Proto](https://github.com/evo-design/proto-tools).

## Available tools

In [2]:
display_available_tools("random_protein")

- **`run_random_protein_sample()`** — Sample protein sequences by filling masked positions with random amino acids drawn from a codon scheme

### `run_random_protein_sample`

`run_random_protein_sample` fills masked positions in protein sequences with amino acids drawn from a codon scheme distribution. Positions marked with `_` are replaced by a randomly sampled amino acid; all other positions are preserved unchanged. The codon scheme controls which amino acids are possible and how frequently each appears: UNIFORM weights all 20 amino acids equally, while library-derived schemes like NNK and NDT reflect the codon redundancy of real degenerate oligonucleotide libraries. The masking strategy system lets you select positions by count, fraction, or index, or you can pre-mask positions manually.

In [3]:
from proto_tools import RandomProteinSampleInput, RandomProteinSampleConfig, run_random_protein_sample
from proto_tools.transforms.masking import MaskingStrategy

In [4]:
# Display input docs
display_api_reference("random_protein", "input", "run_random_protein_sample")

# Underscore characters mark positions for random substitution
inputs = RandomProteinSampleInput(sequences=["MK_AY_AKQR"])

**Input** — `MaskedModelInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Protein sequence(s) to process as string or list of strings. (will be normalized to List[str]) |

In [5]:
# Display config docs
display_api_reference("random_protein", "config", "run_random_protein_sample")

# Use a masking strategy to automatically select 3 positions to mutate | see docs above for all fields
config = RandomProteinSampleConfig(
    masking_strategy=MaskingStrategy(num_mutations=3),
    seed=42,
)

**Config** — `RandomProteinSampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>masking_strategy</code> | <code>proto_tools.transforms.masking.base.RandomMaskingStrategy</code> | <code>RandomMaskingStrategy(method='random', num_mutations=None, mask_fraction=None, fixed_positions=None)</code> | Controls which positions to mask for sampling. Default: random 30%. |
| <code>codon_scheme</code> | <code>Literal['UNIFORM', 'NNN', 'NNK', 'NNS', 'NDT', 'DBK', 'NRT']</code> | <code>'UNIFORM'</code> | Codon scheme for amino acid sampling probabilities. |
| <code>allow_stop_codons</code> | <code>bool</code> | <code>False</code> | Include the stop symbol '*' in the sampling distribution. |
| <code>excluded_amino_acids</code> | <code>list[Literal['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']] &#124; None</code> | <code>None</code> | Residues to remove after codon/stop handling (e.g. ['C'] to forbid cysteine). |

In [6]:
# Run the sampling function
wild_type = "MKTLAYAKQR"
result = run_random_protein_sample(
    RandomProteinSampleInput(sequences=[wild_type]),
    config,
)

In [7]:
# Display output docs
display_api_reference("random_protein", "output", "run_random_protein_sample")

# Show the variant and which positions were mutated
variant = result.sequences[0]
diffs = [i for i, (a, b) in enumerate(zip(wild_type, variant)) if a != b]
print(f"Wild type:         {wild_type}")
print(f"Variant:           {variant}")
print(f"Mutated positions: {diffs}")

# Demonstrate pre-masked input
result_premask = run_random_protein_sample(RandomProteinSampleInput(sequences=["MK_AY_AKQR"]))
print(f"\nPre-masked input:  MK_AY_AKQR")
print(f"Variant:           {result_premask.sequences[0]}")

# Demonstrate codon scheme distributions
print("\nCodon scheme comparison (all-masked sequence):")
for scheme in ["UNIFORM", "NNK", "NDT"]:
    scheme_config = RandomProteinSampleConfig(codon_scheme=scheme, seed=42)
    scheme_result = run_random_protein_sample(
        RandomProteinSampleInput(sequences=["____"]),
        scheme_config,
    )
    print(f"  {scheme:>7s}: {scheme_result.sequences[0]}")

# Demonstrate amino acid exclusions
excluded_config = RandomProteinSampleConfig(excluded_amino_acids=["C"], seed=42)
excluded_result = run_random_protein_sample(
    RandomProteinSampleInput(sequences=["____"]),
    excluded_config,
)
print(f"  no Cys: {excluded_result.sequences[0]}")

**Output** — `RandomProteinSampleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Protein sequences with masked positions randomly filled |

Wild type:         MKTLAYAKQR
Variant:           MKTPAYAAQG
Mutated positions: [3, 7, 9]

Pre-masked input:  MK_AY_AKQR
Variant:           MKRAYIAKQR

Codon scheme comparison (all-masked sequence):
  UNIFORM: PAGF
      NNK: EKSR
      NDT: GNHI
  no Cys: QAHG


## Export Results

Sampling results can be saved to FASTA, TXT, or JSON format for downstream analysis or synthesis ordering.

In [8]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Generate a batch of 5 variants from the same wild-type sequence
batch_inputs = RandomProteinSampleInput(sequences=["MKTLAYAKQR"] * 5)
batch_config = RandomProteinSampleConfig(masking_strategy=MaskingStrategy(num_mutations=2))
batch_result = run_random_protein_sample(batch_inputs, batch_config)

batch_result.export(name="random_protein_variants", export_path=output_dir, file_format="fasta")
print(f"Exported sequences to {output_dir / 'random_protein_variants.fasta'}")

Exported sequences to example_output/random_protein_variants.fasta
